# LAB 6 - Steered MD
**Pulling CLN025 apart**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. Why pull?
2. The coordinate, the spring and the work
3. A box for a stretched peptide
4. The pull code
5. One pull
6. Many pulls at four speeds
7. Force and speed
8. Free energy from non-equilibrium work: Jarzynski's equality
9. Two routes to one free energy
10. Your deliverable

**Learning outcomes:**
- set up a constant-velocity pulling simulation with the GROMACS pull code
- compute the work of a pull and explain why its value depends on the speed
- estimate a free-energy profile from many pulls with Jarzynski's equality, with error bars
- judge when the estimate can be trusted, and compare it with the equilibrium profile of LAB 4

# 1. Why pull?

LAB 4 measured the free-energy profile of CLN025 along its end-to-end distance from unbiased runs, and found it only near the folded state: in nanoseconds the hairpin rarely opens, so the open states were never sampled. **Steered molecular dynamics** (SMD) forces the issue: a spring attached to the peptide is dragged at constant speed, pulling the hairpin open in a few nanoseconds, like the tip of an atomic force microscope or an optical tweezer in a single-molecule experiment.

<img src="imgs/SMD_scheme.png" width="550">

Pulling is a **non-equilibrium** process: the faster the pull, the more of the work is lost as heat. This lab shows how much, and how Jarzynski's equality still recovers the equilibrium free energy from many pulls. The last section compares the result with LAB 4: two independent routes to the same free-energy profile.

## Why the caps

The course's CLN025 carries caps on both ends. With free ends, the charged N- and C-terminus attract each other, and a simulation that pulls CLN025 apart by its ends gives a force that swings between positive and negative values soon after the pull starts (Zhao & Cheng, *Amino Acids* 2012, [doi:10.1007/s00726-011-1150-5](https://doi.org/10.1007/s00726-011-1150-5)). The caps remove those charges, and with them the artefact.

## Hairpins and amyloids

β-hairpins are also the building blocks of amyloid fibrils, the β-sheet aggregates of Alzheimer's and other diseases, whose strands are held together by the same backbone hydrogen bonds you will break here. The optional notebook `../Additional_Material/amyloid-2BEG` pulls one peptide off an Aβ(17–42) fibril (PDB 2BEG) with the same pull code.

# 2. The coordinate, the spring and the work

The **pull coordinate** is the end-to-end distance of LABs 4 and 5: $\xi$, between the Cα atoms of Tyr1 and Tyr10, about 0.5 nm in the folded hairpin. A harmonic spring ties $\xi$ to a position $\lambda$ that moves at constant speed $v$:

$$U_{spring} = \frac{k}{2}\left(\xi - \lambda(t)\right)^2, \qquad \lambda(t) = \lambda_0 + v\,t$$

The spring force on the coordinate is $F = -k(\xi - \lambda)$: positive while the peptide lags behind the spring.

**A stiff spring.** With $k$ = 5000 kJ mol⁻¹ nm⁻², thermal motion stretches the spring by only $\sqrt{k_BT/k} \approx$ 0.02 nm, so $\xi$ stays close to $\lambda$ and the free energy along $\lambda$ approximates the free energy along $\xi$ itself (Park & Schulten, *J Chem Phys* 2004, [doi:10.1063/1.1651473](https://doi.org/10.1063/1.1651473)).

**The work.** Moving the spring costs the work

$$W(t) = v\int_0^t F(t')\,dt'$$

the work of the pulling **protocol**, done by moving $\lambda$. It enters Jarzynski's equality (section 8), and it is plotted against $\lambda$, the position of the spring, not against the measured $\xi$.

# 3. A box for a stretched peptide

The pulled peptide reaches an end-to-end distance above 2 nm, and it can rotate freely. Two conditions set the size of the box:

1. the stretched peptide must stay farther from its periodic images than the 1.0 nm cut-off, in any orientation;
2. GROMACS measures a distance between two atoms by the nearest periodic copy, so a pulled distance must stay below half of the box; GROMACS stops the run above 0.49 times the box size.

A rhombic dodecahedron with 4.8 nm box vectors satisfies both for a pull from about 0.57 to 2.27 nm. It holds about 7,700 atoms, twice the LAB 3 system, so each nanosecond costs about twice as much.

In [ ]:
%%bash
d=smd_model1
mkdir -p $d/{00-build,01-em,02-nvt,03-npt,04-eq,05-pull}
cp ../common/mdp/*.mdp $d/
python3 ../common/scripts/build_caps.py ../common/structures/2RVD.pdb 1 $d/00-build/cln025_capped.pdb > /dev/null
cd $d/00-build
gmx pdb2gmx -f cln025_capped.pdb -o cln025.gro -p topol.top -i posre.itp -ff amber99sb-ildn -water tip3p -ignh > pdb2gmx.out 2>&1 || tail -n 20 pdb2gmx.out
gmx editconf -f cln025.gro -o box.gro -c -box 4.8 -bt dodecahedron > editconf.out 2>&1 || tail -n 20 editconf.out
gmx solvate -cp box.gro -cs spc216.gro -o solvated.gro -p topol.top > solvate.out 2>&1 || tail -n 20 solvate.out
gmx grompp -f ../em.mdp -c solvated.gro -p topol.top -o ions.tpr -maxwarn 1 > grompp_ions.out 2>&1 || tail -n 20 grompp_ions.out
echo SOL | gmx genion -s ions.tpr -o system.gro -p topol.top -pname NA -nname CL -neutral -conc 0.15 > genion.out 2>&1 || tail -n 20 genion.out
tail -n 4 topol.top
echo "atoms: $(sed -n 2p system.gro)"

`editconf -box 4.8` sets the length of the box vectors directly, instead of a distance from the peptide (`-d`). Minimisation and the two equilibration stages are those of LAB 3, with the course's parameter files. They take a while: about twice the LAB 3 equilibration.

In [ ]:
%%bash
cd smd_model1/01-em
gmx grompp -f ../em.mdp -c ../00-build/system.gro -p ../00-build/topol.top -o em.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm em -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep "converged" em.log
cd ../02-nvt
gmx grompp -f ../nvt.mdp -c ../01-em/em.gro -r ../01-em/em.gro -p ../00-build/topol.top -o nvt.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm nvt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
cd ../03-npt
gmx grompp -f ../npt.mdp -c ../02-nvt/nvt.gro -r ../02-nvt/nvt.gro -t ../02-nvt/nvt.cpt -p ../00-build/topol.top -o npt.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm npt -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
grep -B 1 "Performance:" npt.log

# 4. The pull code

The pull code needs two groups of atoms, here one atom each: an index file names them.

In [ ]:
%%bash
cd smd_model1/00-build
gmx select -s system.gro -on pull.ndx -select '"System" all' '"Tyr1_CA" resid 1 and name CA' '"Tyr10_CA" resid 10 and name CA' > select.out 2>&1 || tail -n 20 select.out
cat pull.ndx | head -n 4
grep -A 1 "Tyr" pull.ndx

With `-n`, `grompp` knows only the groups of the index file, so it also holds `System` for the thermostat. The pulling section is appended to the production parameters of LAB 3:

In [ ]:
%%bash
cd smd_model1
cat md.mdp - > pull.mdp << 'MDP'

; Pulling: distance between the CA atoms of Tyr1 and Tyr10, stiff spring
pull                    = yes
pull-ngroups            = 2
pull-ncoords            = 1
pull-group1-name        = Tyr1_CA
pull-group2-name        = Tyr10_CA
pull-coord1-type        = umbrella
pull-coord1-geometry    = distance
pull-coord1-groups      = 1 2
pull-coord1-dim         = Y Y Y
pull-coord1-start       = yes      ; the spring starts at the current distance
pull-coord1-init        = 0
pull-coord1-rate        = 0        ; nm/ps; 0 = spring at rest
pull-coord1-k           = 5000     ; kJ mol-1 nm-2
pull-nstxout            = 50       ; distance every 0.1 ps
pull-nstfout            = 50       ; force every 0.1 ps
pull-print-ref-value    = yes      ; also write the position of the spring
MDP
tail -n 18 pull.mdp

| Option | Meaning |
|---|---|
| `pull-coord1-type = umbrella` | A harmonic spring (as opposed to a rigid constraint) |
| `pull-coord1-geometry = distance`, `dim = Y Y Y` | The coordinate is the full 3D distance between the two groups |
| `pull-coord1-k` | The spring constant |
| `pull-coord1-rate` | The speed of the spring, $v$; 0 holds it still |
| `pull-coord1-start = yes`, `init = 0` | The spring starts where the distance is at the start of the run |
| `pull-nstxout`, `pull-nstfout` | How often $\xi$ (`_pullx.xvg`) and $F$ (`_pullf.xvg`) are written |
| `pull-print-ref-value` | Also write $\lambda$ in `_pullx.xvg` |

## 4.1 Starting structures

Jarzynski's equality needs pulls that start from **equilibrium**, with the spring at its starting position. The next run holds the spring still for 1 ns (`rate = 0`), and 20 structures are taken from it, one every 50 ps. The rest position of the spring is saved: every pull starts from it.

In [ ]:
%%bash
cd smd_model1/04-eq
gmx grompp -f ../pull.mdp -c ../03-npt/npt.gro -t ../03-npt/npt.cpt -n ../00-build/pull.ndx -p ../00-build/topol.top -o eq.tpr > grompp.out 2>&1 || tail -n 20 grompp.out
gmx mdrun -deffnm eq -ntmpi 1 > mdrun.out 2>&1 || tail -n 20 mdrun.out
echo System | gmx trjconv -s eq.tpr -f eq.xtc -o start.gro -sep -b 50 -skip 5 > trjconv.out 2>&1 || tail -n 20 trjconv.out
grep -v '^[@#]' eq_pullx.xvg | head -n 1 | awk '{print $3}' > rest_position.txt
echo "$(ls start*.gro | wc -l) starting structures; rest position of the spring: $(cat rest_position.txt) nm"

# 5. One pull

Every pull is 1.7 nm long, from the rest position, with new velocities at 300 K. This function writes the parameter file of one pull and its run input file:

In [ ]:
import re
import subprocess

rest = float(open("smd_model1/04-eq/rest_position.txt").read())
pull_mdp = open("smd_model1/pull.mdp").read()


def prepare_pull(speed, start):
    """Parameter and run input file of one pull at `speed` nm/ps from starting structure `start` (0-19)."""
    folder = f"smd_model1/05-pull/v{speed}"
    name = f"pull_{start + 1:02d}"
    subprocess.run(["mkdir", "-p", folder])
    mdp = pull_mdp
    for key, value in [("pull-coord1-rate", speed), ("nsteps", round(1.7 / speed / 0.002)),
                       ("pull-coord1-start", "no       ; the spring starts at its rest position"),
                       ("pull-coord1-init", rest), ("continuation", "no"),
                       ("gen-vel", "yes\ngen-temp        = 300\ngen-seed        = -1")]:
        mdp = re.sub(rf"(?m)^{key} .*", f"{key:<23} = {value}", mdp)
    with open(f"{folder}/{name}.mdp", "w") as f:
        f.write(mdp)
    command = (f"cd {folder} && gmx grompp -f {name}.mdp -c ../../04-eq/start{start}.gro -n ../../00-build/pull.ndx "
               f"-p ../../00-build/topol.top -o {name}.tpr > {name}.grompp.out 2>&1")
    result = subprocess.run(command, shell=True)
    return f"{folder}/{name}", result.returncode


print(prepare_pull(0.05, 0))

The fastest pull, 0.05 nm/ps, takes 34 ps: run it here.

In [ ]:
%%bash
cd smd_model1/05-pull/v0.05
gmx mdrun -deffnm pull_01 -ntmpi 1 > pull_01.mdrun.out 2>&1 || tail -n 20 pull_01.mdrun.out
head -n 40 pull_01_pullx.xvg | grep legend

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append("../common/scripts")
import cln025
from cln025 import xvg

KT = cln025.KT
TO_PN = 1.66054                                    # 1 kJ mol-1 nm-1 = 1.66 pN


def load_pull(prefix, speed):
    """Time, distance, spring position, force and protocol work of one pull."""
    x, f = xvg(prefix + "_pullx.xvg"), xvg(prefix + "_pullf.xvg")
    n = min(len(x), len(f))
    t, xi, lam, F = x[:n, 0], x[:n, 1], x[:n, 2], f[:n, 1]
    W = speed * np.concatenate([[0], np.cumsum(0.5 * (F[1:] + F[:-1]) * np.diff(t))])
    return {"t": t, "xi": xi, "lam": lam, "F": F, "W": W}


p = load_pull("smd_model1/05-pull/v0.05/pull_01", 0.05)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(p["t"], p["lam"], label="spring λ")
axes[0].plot(p["t"], p["xi"], label="end-to-end ξ")
axes[0].set_xlabel("time (ps)")
axes[0].set_ylabel("distance (nm)")
axes[0].legend()
axes[1].plot(p["lam"], p["F"] * TO_PN)
axes[1].set_xlabel("spring position λ (nm)")
axes[1].set_ylabel("force (pN)")
axes[2].plot(p["lam"], p["W"])
axes[2].set_xlabel("spring position λ (nm)")
axes[2].set_ylabel("work W (kJ/mol)")
fig.tight_layout()
print(f"largest force {p['F'].max() * TO_PN:.0f} pN, total work {p['W'][-1]:.0f} kJ/mol = {p['W'][-1] / KT:.0f} kT")

<div class="alert alert-block alert-info">How far does the peptide lag behind the spring? Where does the force rise, and what happens in the peptide there (open the trajectory, <code>pull_01.xtc</code>, in VMD with <code>smd_model1/00-build/system.gro</code>)? One pull gives one value of the work, and cannot give a free energy: section 8 explains why.</div>

# 6. Many pulls at four speeds

The course's set is 60 pulls:

| Speed (nm/ps) | Pulls | Length of one pull | Total |
|---|---|---|---|
| 0.05 | 20 | 34 ps | 0.7 ns |
| 0.01 | 20 | 170 ps | 3.4 ns |
| 0.005 | 10 | 340 ps | 3.4 ns |
| 0.001 | 10 | 1.7 ns | 17 ns |

Prepare them all; then run your share. The next cell writes a script that runs a list of pulls one after the other:

In [ ]:
plan = {0.05: range(20), 0.01: range(20), 0.005: range(10), 0.001: range(10)}
for speed, starts in plan.items():
    for start in starts:
        prefix, status = prepare_pull(speed, start)
        if status:
            print("grompp failed for", prefix)

with open("run_pulls.sh", "w") as f:
    f.write("#!/bin/bash\n# Usage: bash run_pulls.sh SPEED FIRST LAST   e.g. bash run_pulls.sh 0.01 1 20\n")
    f.write("cd smd_model1/05-pull/v$1\n")
    f.write("for i in $(seq -f %02g $2 $3); do\n")
    f.write("    [ -f pull_${i}_pullf.xvg ] && [ -f pull_$i.gro ] && continue\n")
    f.write("    gmx mdrun -deffnm pull_$i -ntmpi 1 > pull_$i.mdrun.out 2>&1\n    echo \"pull $i at $1 nm/ps done\"\ndone\n")
print(open("run_pulls.sh").read())

Run the 20 fast pulls and the 20 at 0.01 nm/ps from a terminal (about 4 ns in all), in the background:

```bash
cd ~/molbiomech/labs/06-SteeredMD
nohup bash -c "bash run_pulls.sh 0.05 1 20; bash run_pulls.sh 0.01 1 20" > run_pulls.out 2>&1 &
```

The slow pulls are shared among the class, or downloaded: `course-fetch --lab 06-SteeredMD --solutions` puts the course's 60 pulls (their `_pullx.xvg` and `_pullf.xvg` files) in `solutions/v<speed>/`. Choose which set the analysis reads:

In [ ]:
%env PULLS=solutions

`solutions` for the course's pulls, `smd_model1/05-pull` for yours.

In [ ]:
import glob
import os

speeds = [0.001, 0.005, 0.01, 0.05]
pulls = {}
for speed in speeds:
    prefixes = sorted(p[:-len("_pullf.xvg")] for p in glob.glob(f"{os.environ['PULLS']}/v{speed}/pull_*_pullf.xvg"))
    pulls[speed] = [load_pull(prefix, speed) for prefix in prefixes]
    n = min((len(p["t"]) for p in pulls[speed]), default=0)
    for p in pulls[speed]:
        for key in p:
            p[key] = p[key][:n]
    print(f"{speed} nm/ps: {len(pulls[speed])} pulls")

# 7. Force and speed

In [ ]:
fig, axes = plt.subplots(1, len(speeds), figsize=(16, 3.8), sharey=True)
peak = {}
for ax, speed in zip(axes, speeds):
    peak[speed] = []
    for p in pulls[speed]:
        window = max(1, int(round(0.02 / speed / 0.1)))         # smooth over 0.02 nm of spring motion
        smooth = np.convolve(p["F"], np.ones(window) / window, mode="same")
        ax.plot(p["lam"], smooth * TO_PN, lw=0.5, color="gray")
        peak[speed].append(smooth.max() * TO_PN)
    if pulls[speed]:
        mean_force = np.mean([p["F"] for p in pulls[speed]], axis=0)
        ax.plot(pulls[speed][0]["lam"], mean_force * TO_PN, color="red", label="mean")
    ax.set_title(f"{speed} nm/ps")
    ax.set_xlabel("spring position λ (nm)")
axes[0].set_ylabel("force (pN)")
axes[0].legend()
fig.tight_layout()

fig, ax = plt.subplots(figsize=(5, 4))
for speed in speeds:
    if peak[speed]:
        ax.errorbar(speed, np.mean(peak[speed]), yerr=np.std(peak[speed], ddof=1), fmt="o", color="black", capsize=3)
ax.set_xscale("log")
ax.set_xlabel("pulling speed (nm/ps)")
ax.set_ylabel("largest force (pN)")
fig.tight_layout()

<div class="alert alert-block alert-info">How does the largest force depend on the speed? In single-molecule experiments the rupture force grows with the logarithm of the loading rate (speed × spring constant). Your slowest pull, 0.001 nm/ps, is 1 m/s; an atomic force microscope pulls at about a micrometre per second, a million times slower. What would you expect its forces to be? (No force measurement exists for chignolin, so the comparison stops here.)</div>

# 8. Free energy from non-equilibrium work: Jarzynski's equality

Jarzynski's equality (*Phys Rev Lett* 1997, [doi:10.1103/PhysRevLett.78.2690](https://doi.org/10.1103/PhysRevLett.78.2690)) connects the work of non-equilibrium pulls, started from equilibrium, to the equilibrium free-energy difference:

$$\left\langle e^{-\beta W(\lambda)} \right\rangle = e^{-\beta\, \Delta G(\lambda)} \quad\Longleftrightarrow\quad \Delta G(\lambda) = -\frac{1}{\beta} \ln \left\langle e^{-\beta W(\lambda)} \right\rangle, \qquad \beta = \frac{1}{k_B T}$$

The average runs over many pulls, at the same spring position $\lambda$. Two approximations are also useful:

- the **mean work** $\langle W\rangle$, an upper bound: $\Delta G \le \langle W\rangle$, and the difference is the work dissipated as heat;
- the **second-order cumulant expansion**, $\Delta G \approx \langle W\rangle - \frac{\beta}{2}\sigma_W^2$, exact when the work is Gaussian, as it is for a stiff spring and not too fast a pull (Park & Schulten 2004).

With one pull, $-\frac{1}{\beta}\ln e^{-\beta W} = W$, whatever $\beta$: a single pull gives the work, not the free energy.

In [ ]:
beta = 1 / KT                                   # KT in kJ/mol: work and free energy in kJ/mol


def jarzynski(W):
    """Exponential average over pulls (rows of W), at each spring position."""
    w_min = W.min(axis=0)                        # factor out the smallest work: exp(-beta W) would underflow
    return w_min - np.log(np.mean(np.exp(-beta * (W - w_min)), axis=0)) / beta


def cumulant2(W):
    return W.mean(axis=0) - beta / 2 * W.var(axis=0, ddof=1)


def bootstrap(estimator, W, n_boot=500, seed=1):
    """Standard deviation of the estimator over pulls drawn with repetition."""
    rng = np.random.default_rng(seed)
    samples = [estimator(W[rng.integers(0, len(W), len(W))]) for _ in range(n_boot)]
    return np.std(samples, axis=0)


results = {}
fig, axes = plt.subplots(1, len(speeds), figsize=(16, 4), sharey=True)
for ax, speed in zip(axes, speeds):
    if len(pulls[speed]) < 2:
        continue
    W = np.array([p["W"] for p in pulls[speed]])
    lam = pulls[speed][0]["lam"]
    dG, err = jarzynski(W), bootstrap(jarzynski, W)
    results[speed] = (lam, dG, err)
    for w in W:
        ax.plot(lam, w, color="gray", lw=0.4)
    ax.plot(lam, W.mean(axis=0), color="blue", label="⟨W⟩")
    ax.plot(lam, cumulant2(W), color="green", label="2nd-order cumulant")
    ax.fill_between(lam, dG - err, dG + err, color="red", alpha=0.3)
    ax.plot(lam, dG, color="red", label="Jarzynski")
    ax.set_title(f"{speed} nm/ps, {len(W)} pulls")
    ax.set_xlabel("spring position λ (nm)")
    print(f"{speed:>6} nm/ps: at λ = {lam[-1]:.2f} nm, <W> = {W[:, -1].mean():6.1f}, "
          f"Jarzynski ΔG = {dG[-1]:6.1f} ± {err[-1]:4.1f}, spread of W = {W[:, -1].std(ddof=1) / KT:4.1f} kT")
axes[0].set_ylabel("kJ/mol")
axes[0].legend()
fig.tight_layout()

<div class="alert alert-block alert-info">
<ul>
<li>How does the dissipated work, ⟨W⟩ − ΔG, change with the speed?</li>
<li>The exponential average is dominated by the rare pulls with the <i>lowest</i> work. When the spread of the work is many $k_BT$, a handful of pulls cannot sample those rare ones, and the estimate just follows the lowest-work pull. At which speeds do you trust it?</li>
<li>Where do the three estimates agree, and where do they part?</li>
</ul>
</div>

## 8.1 The work distribution

At the end of the pull, compare the histogram of the work with ⟨W⟩ and ΔG:

In [ ]:
fig, axes = plt.subplots(1, len(speeds), figsize=(16, 3.5))
for ax, speed in zip(axes, speeds):
    if speed not in results:
        continue
    final = np.array([p["W"][-1] for p in pulls[speed]])
    ax.hist(final, bins="auto", color="gray", edgecolor="black")
    ax.axvline(final.mean(), color="blue", label="⟨W⟩")
    ax.axvline(results[speed][1][-1], color="red", label="ΔG (Jarzynski)")
    ax.set_title(f"{speed} nm/ps")
    ax.set_xlabel("W at the end (kJ/mol)")
axes[0].set_ylabel("pulls")
axes[0].legend()
fig.tight_layout()

# 9. Two routes to one free energy

LAB 4 computed the free-energy profile along the same coordinate from unbiased equilibrium runs, $F(d) = -k_BT \ln P(d)$, and saved it in `../04-Analysis/results/pmf_e2e_amber_300K.dat`. With a stiff spring, the Jarzynski profile along $\lambda$ estimates the same profile. Both are defined up to a constant: shift the equilibrium profile to zero at the rest position of the spring, where every Jarzynski profile starts.

In [ ]:
path = "../04-Analysis/results/pmf_e2e_amber_300K.dat"                  # yours, from LAB 4 section 11
if not os.path.exists(path):
    path = "../04-Analysis/solutions/results/pmf_e2e_amber_300K.dat"    # the course's
if not os.path.exists(path):
    print("Run section 11 of LAB 4, or fetch its solutions: course-fetch --lab 04-Analysis --solutions")
else:
    d, F, F_err, frames = np.loadtxt(path, unpack=True)
    start = next(iter(results.values()))[0][0]            # rest position of the spring in the pulls analysed
    F = F - np.interp(start, d, F)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(d, F, yerr=F_err, fmt="o", color="black", ms=3, capsize=2, label="equilibrium, LAB 4")
    for speed, color in zip(speeds, ["tab:red", "tab:orange", "tab:green", "tab:blue"]):
        if speed in results:
            lam, dG, err = results[speed]
            ax.plot(lam, dG, color=color, label=f"Jarzynski, {speed} nm/ps")
            ax.fill_between(lam, dG - err, dG + err, color=color, alpha=0.2)
    ax.set_xlim(0.4, 2.3)
    ax.set_ylim(-5, 80)
    ax.set_xlabel("end-to-end distance / spring position (nm)")
    ax.set_ylabel("free energy (kJ/mol)")
    ax.legend()
    fig.tight_layout()
    fig.savefig("capstone.png", dpi=300)

**What to expect.** The two routes start together at the rest position of the spring, then separate: every speed rises above the equilibrium profile, the slowest least. Jarzynski's average is dominated by the rare pulls of lowest work, and with ten to twenty pulls per speed those are missing (section 8), so the estimate stays above the true profile and drifts further up the faster the pull. That ordering, not a number, is the result of this lab.

<div class="alert alert-block alert-info">
<ul>
<li>Over which range can the two routes be compared at all, and why is the equilibrium profile so short?</li>
<li>Where they overlap, how far does each speed follow the equilibrium profile before it leaves it? Why does the estimate get worse as the pull gets faster?</li>
<li>Beyond the range of LAB 4, the Jarzynski profile is the only estimate you have. How far would you trust it, given section 8?</li>
<li>Both profiles use the same force field, amber99sb-ildn, which overstabilises the folded hairpin (LAB 4): would they agree with an experiment?</li>
</ul>
</div>

# 10. Your deliverable

1. **The force curves** of section 7 at the four speeds, and the largest force against the speed.
2. **The work distributions** of section 8.1, with ⟨W⟩ and ΔG marked, and the dissipated work against the speed.
3. **The overlay** of section 9: the Jarzynski profiles with their bootstrap error bars on the equilibrium profile of LAB 4.
4. **A written discussion**: where and why the two estimates agree or part, and how many pulls, at which speed, you would need to trust the profile of the open hairpin.

As for the other simulation labs, present them in a few slides during a 15-minute group discussion.